<a href="https://colab.research.google.com/github/nikenyudha/Learn_SQL/blob/main/SQLBASIC_NULL_Handling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##USE SQLLITE LIBRARY AND TEST

In [1]:
# Import library
import sqlite3

# Connect to an SQLite database; use ':memory:' for an in-memory database
conn = sqlite3.connect(':memory:')

`NULL` means unknown or absent. It is not zero. It is not an empty string. It is the absence of a value. This distinction matters most when you try to compare it: any comparison with NULL returns NULL (unknown), not TRUE or FALSE. This is called three-value logic.

In [ ]:
c = conn.cursor()
c.execute('''
          CREATE TABLE employees (
                 id INTEGER PRIMARY KEY,
                 name TEXT NOT NULL,
                 salary REAL,
                 manager TEXT)
          ''')

In [ ]:
c.execute("INSERT INTO employees VALUES (1, 'Alice',   95000, 'Carol')")
c.execute("INSERT INTO employees VALUES (2, 'Bob',     72000, 'Carol')")
c.execute("INSERT INTO employees VALUES (3, 'Carol',  130000, NULL)")
c.execute("INSERT INTO employees VALUES (4, 'David',   68000, 'Alice')")
c.execute("INSERT INTO employees VALUES (5, 'Eva',     NULL,  'Alice')")

In [ ]:
#NULL = NULL is NOT true — it returns NULL
c.execute("SELECT name, salary, salary = NULL       AS eq_null, salary IS NULL      AS is_null, salary IS NOT NULL  AS has_salary FROM employees")
print(c.fetchall())

[('Alice', 95000.0, None, 0, 1), ('Bob', 72000.0, None, 0, 1), ('Carol', 130000.0, None, 0, 1), ('David', 68000.0, None, 0, 1), ('Eva', None, None, 1, 0)]


In [ ]:
import pandas as pd
df = pd.read_sql_query("SELECT name, salary, salary = NULL       AS eq_null, salary IS NULL      AS is_null, salary IS NOT NULL  AS has_salary FROM employees", conn)
print(df)

    name    salary eq_null  is_null  has_salary
0  Alice   95000.0    None        0           1
1    Bob   72000.0    None        0           1
2  Carol  130000.0    None        0           1
3  David   68000.0    None        0           1
4    Eva       NaN    None        1           0


Notice that salary = NULL always produces NULL, never TRUE. The only correct way to test for a missing value is IS NULL or IS NOT NULL. This is why a WHERE salary = NULL clause will match zero rows even when nulls exist — you must write WHERE salary IS NULL.

***NULL in Arithmetic and Aggregates***.
<br>
Any arithmetic operation that involves NULL produces NULL. NULL + 1 is NULL. NULL * 100 is NULL. This means a single missing value can silently propagate through calculations and produce unexpected results

In [2]:
c = conn.cursor()
c.execute('''
          CREATE TABLE sales (
                 id   INTEGER PRIMARY KEY,
                 rep  TEXT NOT NULL,
                 base_salary REAL,
                 commission  REAL,
                 bonus       REAL)
          ''')

In [3]:
c.execute("INSERT INTO sales VALUES (1, 'Alice',  60000, 12000, 3000)")
c.execute("INSERT INTO sales VALUES (2, 'Bob',   55000, 8500,  NULL)")
c.execute("INSERT INTO sales VALUES (3, 'Carol',65000, NULL,  2500)")
c.execute("INSERT INTO sales VALUES (4, 'David',  50000, NULL,  NULL)")
c.execute("INSERT INTO sales VALUES (5, 'Eva',    60000, 9000,  1500)")

In [8]:
c.execute("SELECT rep, base_salary, commission, bonus,base_salary + commission + bonus AS total_pay, COUNT(*) AS total_reps,COUNT(commission) AS reps_with_commission, AVG(commission) AS avg_commission,SUM(bonus) AS total_bonus  FROM sales GROUP BY rep, base_salary, commission, bonus")
print(c.fetchall())

[('Alice', 60000.0, 12000.0, 3000.0, 75000.0, 1, 1, 12000.0, 3000.0), ('Bob', 55000.0, 8500.0, None, None, 1, 1, 8500.0, None), ('Carol', 65000.0, None, 2500.0, None, 1, 0, None, 2500.0), ('David', 50000.0, None, None, None, 1, 0, None, None), ('Eva', 60000.0, 9000.0, 1500.0, 70500.0, 1, 1, 9000.0, 1500.0)]


In [9]:
import pandas as pd
df2 = pd.read_sql_query("SELECT rep, base_salary, commission, bonus,base_salary + commission + bonus AS total_pay, COUNT(*) AS total_reps,COUNT(commission) AS reps_with_commission, AVG(commission) AS avg_commission,SUM(bonus) AS total_bonus  FROM sales GROUP BY rep, base_salary, commission, bonus", conn)
print(df2)

     rep  base_salary  commission   bonus  total_pay  total_reps  \
0  Alice      60000.0     12000.0  3000.0    75000.0           1   
1    Bob      55000.0      8500.0     NaN        NaN           1   
2  Carol      65000.0         NaN  2500.0        NaN           1   
3  David      50000.0         NaN     NaN        NaN           1   
4    Eva      60000.0      9000.0  1500.0    70500.0           1   

   reps_with_commission  avg_commission  total_bonus  
0                     1         12000.0       3000.0  
1                     1          8500.0          NaN  
2                     0             NaN       2500.0  
3                     0             NaN          NaN  
4                     1          9000.0       1500.0  


Two key rules to internalize here:

- Arithmetic with NULL gives NULL. base_salary + commission + bonus is NULL whenever any part is missing.
- Aggregate functions ignore NULLs. COUNT(commission) counts only the rows where commission is not null. AVG(commission) divides the sum by the number of non-null rows, not the total row count. This is usually what you want — but it can surprise you when COUNT(*) and COUNT(column) return different numbers.

### COALESCE: Substituting a Default for NULL

`COALESCE(expr1, expr2, ...)` returns the first non-null value from its argument list. It is the standard way to replace a missing value with a sensible default. You can pass as many arguments as needed; COALESCE walks through them left to right and returns the first one that isn't NULL.

In [11]:
c = conn.cursor()
c.execute('''
          CREATE TABLE sales_reps(
                 id          INTEGER PRIMARY KEY,
                 name        TEXT NOT NULL,
                 region      TEXT,
                 commission  REAL,
                 bonus       REAL)
          ''')

In [12]:
c.execute("INSERT INTO sales_reps VALUES (1, 'Alice',  'North', 12000, 3000)")
c.execute("INSERT INTO sales_reps VALUES (2, 'Bob',    'South', 8500,  NULL)")
c.execute("INSERT INTO sales_reps VALUES (3, 'Carol',  NULL,    NULL,  2500)")
c.execute("INSERT INTO sales_reps VALUES (4, 'David',  'East',  NULL,  NULL)")
c.execute("INSERT INTO sales_reps VALUES (5, 'Eva',    'West',  9000,  1500)")

In [13]:
#Replace NULL region with a literal default(COALESCE(region, 'Unassigned') AS region)
#Replace NULL commission with 0 before adding (COALESCE(commission, 0) + COALESCE(bonus, 0)  AS total_extra)
#Chain of fallbacks: prefer commission, then bonus, then 0 (COALESCE(commission, bonus, 0) AS primary_extra)
c.execute("SELECT name, COALESCE(region, 'Unassigned') AS region, COALESCE(commission, 0) + COALESCE(bonus, 0)  AS total_extra,COALESCE(commission, bonus, 0) AS primary_extra FROM sales_reps ORDER BY name")
print(c.fetchall())

[('Alice', 'North', 15000.0, 12000.0), ('Bob', 'South', 8500.0, 8500.0), ('Carol', 'Unassigned', 2500.0, 2500.0), ('David', 'East', 0, 0), ('Eva', 'West', 10500.0, 9000.0)]


In [14]:
import pandas as pd
dfcoalesce = pd.read_sql_query("SELECT name, COALESCE(region, 'Unassigned') AS region, COALESCE(commission, 0) + COALESCE(bonus, 0)  AS total_extra,COALESCE(commission, bonus, 0) AS primary_extra FROM sales_reps ORDER BY name", conn)
print(dfcoalesce)

    name      region  total_extra  primary_extra
0  Alice       North      15000.0        12000.0
1    Bob       South       8500.0         8500.0
2  Carol  Unassigned       2500.0         2500.0
3  David        East          0.0            0.0
4    Eva        West      10500.0         9000.0


`COALESCE` is especially useful in arithmetic. Instead of letting one NULL poison an entire calculation, wrap each nullable column: `COALESCE(commission, 0) + COALESCE(bonus, 0)` always returns a number. The chained form — `COALESCE(commission, bonus, 0)` — is handy when you have a priority order: use the first available value, fall back to the next, and ultimately default to zero.

### NULLIF: Turning Values into NULL on Purpose

`NULLIF(expr, value)` does the reverse of `COALESCE`. It returns NULL if expr equals value, otherwise it returns expr unchanged. This is useful when you have a sentinel value — like 0, 'N/A', or 'unknown' — that was used to represent missing data but should be treated as NULL in your queries.

In [16]:
# means they skipped the question
# 'N/A' means uncategorized
c = conn.cursor()
c.execute('''
          CREATE TABLE survey(
                id        INTEGER PRIMARY KEY,
                respondent TEXT NOT NULL,
                score     INTEGER,
                category  TEXT)
          ''')

In [17]:
c.execute("INSERT INTO survey VALUES (1, 'Alice', 8,  'Product')")
c.execute("INSERT INTO survey VALUES (2, 'Bob',   0,  'Service')")
c.execute("INSERT INTO survey VALUES (3, 'Carol', 7,  'N/A')")
c.execute("INSERT INTO survey VALUES (4, 'David', 0,  'N/A')")
c.execute("INSERT INTO survey VALUES (5, 'Eva',   9,  'Product')")
c.execute("INSERT INTO survey VALUES (6, 'Frank', 6,  'Service')")

In [18]:
#Treat score of 0 as NULL (skipped)
#Treat 'N/A' category as NULL
# Average only over genuine responses
c.execute("SELECT respondent, score, NULLIF(score, 0) AS real_score, category, NULLIF(category, 'N/A')AS real_category, AVG(NULLIF(score, 0)) OVER () AS avg_real_score FROM survey ")
print(c.fetchall())

[('Alice', 8, 8, 'Product', 'Product', 7.5), ('Bob', 0, None, 'Service', 'Service', 7.5), ('Carol', 7, 7, 'N/A', None, 7.5), ('David', 0, None, 'N/A', None, 7.5), ('Eva', 9, 9, 'Product', 'Product', 7.5), ('Frank', 6, 6, 'Service', 'Service', 7.5)]


In [20]:
import pandas as pd
dfcnullif = pd.read_sql_query("SELECT respondent, score, NULLIF(score, 0) AS real_score, category, NULLIF(category, 'N/A')AS real_category, AVG(NULLIF(score, 0)) OVER () AS avg_real_score FROM survey", conn)
print(dfcnullif)

  respondent  score  real_score category real_category  avg_real_score
0      Alice      8         8.0  Product       Product             7.5
1        Bob      0         NaN  Service       Service             7.5
2      Carol      7         7.0      N/A          None             7.5
3      David      0         NaN      N/A          None             7.5
4        Eva      9         9.0  Product       Product             7.5
5      Frank      6         6.0  Service       Service             7.5


Without NULLIF, AVG(score) would include the zeros and report a misleadingly low average. By converting zeros to NULL first, the aggregate ignores them automatically. NULLIF is also the standard way to prevent division-by-zero errors: total / NULLIF(count, 0) returns NULL instead of crashing when count is zero.

### Challenge
A customer database has optional phone numbers and address fields. Some rows have empty strings instead of proper NULL because of a legacy import. Fix the data in your query and generate a clean report.

In [21]:
c = conn.cursor()
c.execute('''
          CREATE TABLE customers (
                id      INTEGER PRIMARY KEY,
                name    TEXT NOT NULL,
                email   TEXT NOT NULL,
                phone   TEXT,
                city    TEXT,
                country TEXT)
          ''')

In [25]:
c.execute("INSERT INTO customers VALUES (1, 'Alice Johnson', 'alice@example.com', '555-1234',  'New York',  'US')")
c.execute("INSERT INTO customers VALUES (2, 'Bob Smith',     'bob@example.com',   '',          'London',    'UK')")
c.execute("INSERT INTO customers VALUES (3, 'Carol Lee',     'carol@example.com', NULL,        '',          'CA')")
c.execute("INSERT INTO customers VALUES (4, 'David Kim',     'david@example.com', '555-9876',  NULL,        'US')")
c.execute("INSERT INTO customers VALUES (5, 'Eva Rossi',     'eva@example.com',   '',          'Rome',      NULL)")

In [28]:
import pandas as pd
dfall= pd.read_sql_query("SELECT * FROM customers", conn)
print(dfall)

   id           name              email     phone      city country
0   1  Alice Johnson  alice@example.com  555-1234  New York      US
1   2      Bob Smith    bob@example.com              London      UK
2   3      Carol Lee  carol@example.com      None                CA
3   4      David Kim  david@example.com  555-9876      None      US
4   5      Eva Rossi    eva@example.com                Rome    None


In [26]:
#challenge 1: Treat empty string phone numbers as NULL.
#Show name, and a "contact" column that shows the phone or 'No phone on file'.

c.execute("SELECT name, COALESCE(NULLIF(phone, ''), 'No phone on file') AS contact FROM customers ")
print(c.fetchall())

[('Alice Johnson', '555-1234'), ('Bob Smith', 'No phone on file'), ('Carol Lee', 'No phone on file'), ('David Kim', '555-9876'), ('Eva Rossi', 'No phone on file')]


In [27]:
import pandas as pd
dfc1= pd.read_sql_query("SELECT name, COALESCE(NULLIF(phone, ''), 'No phone on file') AS contact FROM customers", conn)
print(dfc1)

            name           contact
0  Alice Johnson          555-1234
1      Bob Smith  No phone on file
2      Carol Lee  No phone on file
3      David Kim          555-9876
4      Eva Rossi  No phone on file


In [31]:
#Challenge 2: Build a location string "City, Country"
#Use COALESCE to replace NULL city or country with 'Unknown'.
import pandas as pd
dfc2= pd.read_sql_query("SELECT name, COALESCE(NULLIF(city, ''), 'Unknown') AS Cust_city, COALESCE(NULLIF(country, 'None'), 'Unknown') AS Cust_country FROM customers", conn)
print(dfc2)

            name Cust_city Cust_country
0  Alice Johnson  New York           US
1      Bob Smith    London           UK
2      Carol Lee   Unknown           CA
3      David Kim   Unknown           US
4      Eva Rossi      Rome      Unknown


In [33]:
#Challenge 3: Count how many customers have a real phone number
#(neither NULL nor empty string)

import pandas as pd
dfc3= pd.read_sql_query("SELECT name, COUNT(phone) AS real_phone FROM customers WHERE phone IS NOT NULL AND phone != '' ", conn)
print(dfc3)

            name  real_phone
0  Alice Johnson           2


In [35]:
dfc3 = pd.read_sql_query("""
    SELECT COUNT(*) AS total_customers
    FROM customers
    WHERE phone IS NOT NULL AND phone != ''
""", conn)
print(dfc3)

   total_customers
0                2


In [34]:
dfc3 = pd.read_sql_query("""
    SELECT name, COUNT(phone) AS real_phone
    FROM customers
    WHERE phone IS NOT NULL AND phone != ''
    GROUP BY name
""", conn)
print(dfc3)

            name  real_phone
0  Alice Johnson           1
1      David Kim           1
